# Bagging Regressor: Diabetes Dataset

## Overview

**Bagging** for regression works similarly to classification: it trains multiple decision tree regressors on bootstrap samples and averages their predictions. This reduces variance and improves prediction accuracy.

## Key Concept: How Bagging Works for Regression

1. **Bootstrap Sampling:** Create M different subsets by sampling with replacement
2. **Train M Regressors:** Each regressor is trained independently on a different subset
3. **Average Predictions:** Final prediction is the mean of all M predictions
4. **Result:** Lower variance than single tree, better generalization

## Learning Objectives

By the end of this notebook, you will understand:
1. How Bagging reduces prediction variance
2. How to train and evaluate a Bagging Regressor
3. The relationship between ensemble size and performance
4. How Bagging compares to single tree regression

## Step 1: Import Libraries

In [ ]:
from matplotlib import pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

## Step 2: Load and Prepare Data

In [ ]:
# Load the Diabetes dataset
diabetes_ds = load_diabetes()
X = diabetes_ds.data
y = diabetes_ds.target

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Features: {X_train.shape[1]}')

## Step 3: Train Single Decision Tree Regressor (Baseline)

In [ ]:
# Single Decision Tree Regressor (baseline)
dt_single = DecisionTreeRegressor(
    criterion='squared_error',
    max_depth=5,
    random_state=42
)
dt_single.fit(X_train, y_train)

# Evaluate
y_pred_single = dt_single.predict(X_test)
mse_single = mean_squared_error(y_test, y_pred_single)
r2_single = r2_score(y_test, y_pred_single)

print(f'Single Tree - MSE: {mse_single:.2f}, R²: {r2_single:.4f}')

## Step 4: Train Bagging Regressor

Bagging trains multiple base regressors on bootstrap samples and averages their predictions. Key parameters:
- **n_estimators:** Number of base regressors (trees) in the ensemble
- **max_samples:** Number of samples to draw for each base regressor
- **bootstrap:** Whether to use bootstrap sampling (with replacement)

In [ ]:
# Bagging Regressor with 10 decision trees
bagging_reg = BaggingRegressor(
    estimator=DecisionTreeRegressor(criterion='squared_error', max_depth=5),
    n_estimators=10,      # Number of trees
    max_samples=1.0,      # Use all samples (with replacement)
    bootstrap=True,       # Use bootstrap sampling
    random_state=42
)

# Train the ensemble
bagging_reg.fit(X_train, y_train)

# Evaluate
y_pred_bagging = bagging_reg.predict(X_test)
mse_bagging = mean_squared_error(y_test, y_pred_bagging)
r2_bagging = r2_score(y_test, y_pred_bagging)

print(f'Bagging (10 trees) - MSE: {mse_bagging:.2f}, R²: {r2_bagging:.4f}')
print(f'Improvement in MSE: {(mse_single - mse_bagging):.2f}')
print(f'Improvement in R²: {(r2_bagging - r2_single):.4f}')

## Step 5: Effect of Ensemble Size

Let's examine how test set performance changes as we increase the number of trees.

In [ ]:
# Test with different numbers of estimators
n_estimators_list = [1, 5, 10, 20, 50, 100]
bagging_mse_scores = []
bagging_r2_scores = []

for n_est in n_estimators_list:
    bag = BaggingRegressor(
        estimator=DecisionTreeRegressor(criterion='squared_error', max_depth=5),
        n_estimators=n_est,
        bootstrap=True,
        random_state=42
    )
    bag.fit(X_train, y_train)
    y_pred = bag.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    bagging_mse_scores.append(mse)
    bagging_r2_scores.append(r2)
    print(f'n_estimators={n_est:3d}: MSE = {mse:6.2f}, R² = {r2:.4f}')

## Step 6: Visualize Performance

In [ ]:
# Plot MSE vs. number of estimators
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# MSE plot
axes[0].plot(n_estimators_list, bagging_mse_scores, 'o-', linewidth=2, markersize=8, label='Bagging')
axes[0].axhline(y=mse_single, color='r', linestyle='--', linewidth=2, label=f'Single Tree ({mse_single:.2f})')
axes[0].set_xlabel('Number of Estimators')
axes[0].set_ylabel('Mean Squared Error')
axes[0].set_title('Bagging: MSE vs. Number of Trees')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# R² plot
axes[1].plot(n_estimators_list, bagging_r2_scores, 'o-', linewidth=2, markersize=8, label='Bagging')
axes[1].axhline(y=r2_single, color='r', linestyle='--', linewidth=2, label=f'Single Tree ({r2_single:.4f})')
axes[1].set_xlabel('Number of Estimators')
axes[1].set_ylabel('R² Score')
axes[1].set_title('Bagging: R² vs. Number of Trees')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 7: Predictions Visualization

In [ ]:
# Use the model with 10 estimators
final_bagging = BaggingRegressor(
    estimator=DecisionTreeRegressor(criterion='squared_error', max_depth=5),
    n_estimators=10,
    bootstrap=True,
    random_state=42
)
final_bagging.fit(X_train, y_train)
y_pred_bagging_final = final_bagging.predict(X_test)

# Plot predictions vs. actual values
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Single tree
axes[0].scatter(y_test, y_pred_single, alpha=0.6, color='blue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Values')
axes[0].set_ylabel('Predicted Values')
axes[0].set_title(f'Single Tree (R² = {r2_single:.3f})')
axes[0].grid(True, alpha=0.3)

# Bagging
axes[1].scatter(y_test, y_pred_bagging_final, alpha=0.6, color='green')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Values')
axes[1].set_ylabel('Predicted Values')
axes[1].set_title(f'Bagging (R² = {r2_bagging:.3f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Key Takeaways

1. **Bagging averages predictions** instead of voting for regression
2. **Variance reduction:** Multiple diverse trees reduce overfitting
3. **Bootstrap sampling:** Each tree trains on a different random sample (with replacement)
4. **Independent training:** All trees are trained independently (can be parallelized)
5. **Performance improvement:** Bagging typically outperforms single tree on test data
6. **Diminishing returns:** Adding more trees shows diminishing performance gains
7. **No training error guarantee:** Unlike boosting, Bagging doesn't necessarily reduce training error